# ELLA VIDEO ENGINE V2 — Kaggle GPU test

دفتر جديد مستقل عن دفاتر Wan القديمة. يستخدم أوزان CogVideoX-Fun 2B InP المفتوحة مع كود VideoX-Fun الرسمي المثبت على commit معروف. شغّل **Run All** بعد اختيار GPU وتفعيل Internet. الاختبار الافتراضي يولّد 5 ثوانٍ من فيديو T2V حقيقي، ثم يفحص وجود حركة فعلية وقابلية فك MP4.

**مهم:** لم يُختبر هذا الدفتر فعليًا على Tesla T4 بعد. اختيار FP16 وCPU offload توافق تصميمي فقط، وليس إثباتًا للنجاح. لا يتصل الدفتر بتطبيق الإنتاج ولا يرفع أي ملف إلى Render/Cloudinary تلقائيًا.

In [ ]:
# إعدادات اختبار اختياريّة — لا تحتاج تعديلها لتشغيل الاختبار الافتراضي.
PROMPT = "A small red kite dances in a steady breeze above a green hillside at golden hour; the camera slowly tracks forward, leaves and grass move naturally, continuous single shot, no text."
DURATION_SECONDS = 5       # جرّب 5 أو 10 أو 20 أو 30 أو 60؛ المدد الأطول تولّد windows متتابعة.
IMAGE_PATH = None          # اختياري: مسار صورة من Kaggle Input لتجربة image-to-video.
WIDTH, HEIGHT = 672, 384   # مضاعفات 32؛ دقة اختبار 16:9 منخفضة نسبيًا لتقليل ضغط الذاكرة.
FPS = 8
STEPS = 20                # زدها لاحقًا لتحسين الجودة على حساب الوقت.
SEED = 42

import os, sys, json, shutil, subprocess, time, importlib.util
from pathlib import Path
WORK = Path('/kaggle/working')
OUTPUT = WORK / 'ella_video_engine_v2.mp4'
print('Config:', json.dumps({'duration': DURATION_SECONDS, 'fps': FPS, 'resolution': f'{WIDTH}x{HEIGHT}', 'image_to_video': bool(IMAGE_PATH)}, ensure_ascii=False))

In [ ]:
# فحص مسبق للبيئة الفعلية: لا نفترض أن Kaggle أعطانا T4.
import torch
if not torch.cuda.is_available():
    raise RuntimeError('CUDA غير متاح. من Notebook Settings اختر GPU ثم أعد Run All.')
if sys.version_info < (3, 10):
    raise RuntimeError(f'Python 3.10+ مطلوب، الموجود {sys.version}')
props = torch.cuda.get_device_properties(0)
cap = torch.cuda.get_device_capability(0)
runtime = {'gpu': props.name, 'compute_capability': f'{cap[0]}.{cap[1]}', 'vram_gib': round(props.total_memory / 1024**3, 2), 'torch': torch.__version__, 'cuda': torch.version.cuda, 'python': sys.version.split()[0], 'working_free_gib': round(shutil.disk_usage(WORK).free / 1024**3, 2)}
print(json.dumps(runtime, indent=2, ensure_ascii=False))
if cap < (7, 0):
    raise RuntimeError('هذه التهيئة تحتاج CUDA Compute Capability 7.0 أو أحدث.')
if shutil.which('ffmpeg') is None or shutil.which('ffprobe') is None:
    raise RuntimeError('ffmpeg وffprobe مطلوبان لفحص MP4؛ لم يجدهما الدفتر.')
if runtime['working_free_gib'] < 15:
    print('تحذير مساحة: تنزيل الوزن قد يحتاج قرابة 13 GiB؛ المساحة الحرة قد لا تكفي بعد تثبيت الاعتماديات.')
if cap == (7, 5):
    print('بطاقة Turing/CC 7.5 كُشفت؛ المحرك سيستخدم FP16 + sequential CPU offload. هذا ليس إعلان نجاح T4 قبل اكتمال التوليد والتحقق.')

## تجهيز المصدر والاعتماديات
يحافظ التثبيت على PyTorch CUDA الذي وفرته Kaggle، ويثبت باقي مكتبات inference فقط. تُضيّق تهيئة الحزم في نسخة VideoX-Fun المنسوخة لتجنب استيراد موديلات غير مستخدمة؛ ملفات CogVideoX الأصلية نفسها لا تتغير.

In [ ]:
ELLA_REPO = WORK / 'Ella2027'
ELLA_BRANCH = 'feature/ella-wan22-engine'
if (ELLA_REPO / '.git').is_dir():
    subprocess.run(['git', '-C', str(ELLA_REPO), 'fetch', 'origin', ELLA_BRANCH, '--depth=1'], check=True)
    subprocess.run(['git', '-C', str(ELLA_REPO), 'reset', '--hard', 'FETCH_HEAD'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth=1', '--branch', ELLA_BRANCH, 'https://github.com/WaelNameh84/Ella2027.git', str(ELLA_REPO)], check=True)
ENGINE_DIR = ELLA_REPO / 'services' / 'ella-video-engine-v2'
if not (ENGINE_DIR / 'engine.py').is_file():
    raise RuntimeError('لم يعثر الدفتر على ملفات محرك V2 في الفرع. تأكد من Internet وأن الفرع المطلوب متاح.')

UPSTREAM = WORK / 'VideoX-Fun-v2-upstream'
UPSTREAM_COMMIT = '98fffa929bb373edcbfa3620e9aa05cd65a53103'
if not (UPSTREAM / '.git').is_dir():
    subprocess.run(['git', 'clone', '--filter=blob:none', '--no-checkout', 'https://github.com/aigc-apps/VideoX-Fun.git', str(UPSTREAM)], check=True)
subprocess.run(['git', '-C', str(UPSTREAM), 'checkout', '--detach', UPSTREAM_COMMIT], check=True)
subprocess.run([sys.executable, str(ENGINE_DIR / 'prepare_upstream.py'), str(UPSTREAM)], check=True)

# لا يوجد torch في requirements-kaggle.txt عمدًا؛ لا نبدل نسخة CUDA التي فُحصت أعلاه.
before_torch = torch.__version__
subprocess.run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--no-cache-dir', '-r', str(ENGINE_DIR / 'requirements-kaggle.txt')], check=True)
if torch.__version__ != before_torch:
    raise RuntimeError('تغير إصدار PyTorch أثناء التثبيت؛ أوقفنا التوليد بدل المتابعة ببيئة CUDA غير متحققة.')
sys.path.insert(0, str(ENGINE_DIR))
sys.path.insert(0, str(UPSTREAM))
print('ELLA V2 source:', ELLA_REPO)
print('VideoX-Fun pinned revision:', subprocess.check_output(['git', '-C', str(UPSTREAM), 'rev-parse', 'HEAD'], text=True).strip())
print('PyTorch preserved:', torch.__version__)

## تنزيل الوزن (قابل للاستكمال)
يستخدم `snapshot_download` إلى مجلد ثابت في `/kaggle/working`; مكتبة Hugging Face تتحقق من القطع المحلية وتستكمل الملفات الناقصة عند إعادة تشغيل الخلية. الأوزان كبيرة، لذا لا تقطع الجلسة عمدًا أثناء التنزيل.

In [ ]:
from huggingface_hub import snapshot_download
MODEL_ID = 'alibaba-pai/CogVideoX-Fun-V1.1-2b-InP'
MODEL_DIR = WORK / 'ella_v2_models' / 'CogVideoX-Fun-V1.1-2b-InP'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = Path(snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(MODEL_DIR),
    allow_patterns=['*.json', '*.safetensors', '*.bin', '*.model', '*.txt', '*.tiktoken', '*.vocab', '*.merges', '*.spm'],
))
required = ['transformer/config.json', 'vae/config.json', 'text_encoder/config.json', 'tokenizer']
missing = [name for name in required if not (MODEL_DIR / name).exists()]
if missing:
    raise RuntimeError(f'بنية مستودع النموذج تختلف عن المتوقع؛ الملفات/المجلدات الناقصة: {missing}. لم يتم بدء التوليد.')
print('Model path:', MODEL_DIR)
print('Model files:', sum(1 for p in MODEL_DIR.rglob('*') if p.is_file()))

## توليد الفيديو والتحقق من MP4
المقاطع تُولّد بالنموذج فعلًا، وتُبنى الاستمرارية من إطارات نهاية المقطع السابق مع تداخل. يتحقق المشغّل من stream/المدة ويفك عينات إطارات؛ إذا كان الناتج ثابتًا أو غير قابل للفك يفشل ولا يعتبره نجاحًا.

In [ ]:
from engine import generate_video
from core import plan_segments
if IMAGE_PATH is not None and not Path(IMAGE_PATH).is_file():
    raise FileNotFoundError(f'IMAGE_PATH لا يشير إلى ملف: {IMAGE_PATH}')
print('Segment plan:', plan_segments(DURATION_SECONDS, FPS))
def show_progress(done, total):
    print(f'المقاطع المكتملة فعليًا: {done}/{total}')
result = generate_video(
    prompt=PROMPT, output_path=OUTPUT, model_dir=MODEL_DIR,
    duration_seconds=DURATION_SECONDS, image_path=IMAGE_PATH,
    width=WIDTH, height=HEIGHT, fps=FPS, steps=STEPS, seed=SEED,
    progress=show_progress,
)
assert OUTPUT.is_file() and OUTPUT.stat().st_size > 0
print('تم إنشاء وفحص MP4 حقيقي:', json.dumps(result, indent=2, ensure_ascii=False))

In [ ]:
from IPython.display import Video, display
display(Video(str(OUTPUT), embed=True, html_attributes='controls loop'))
print('ملف الخرج:', OUTPUT)
print('تنزيل إلى جهازك من قائمة الملفات في Kaggle: ella_video_engine_v2.mp4 وella_video_engine_v2.json')

## حدود هذا الاختبار
هذا تشغيل يدوي مع حفظ محلي داخل جلسة Kaggle فقط. لا توجد اتصالات API أو أسرار أو رفع تلقائي. الإخراج لا يحتوي TTS/موسيقى مولدة/مزامنة شفاه/AI upscaling؛ يمكن إضافة هذه المسارات بعد اختبار مستقل ومتكامل. استيفاء إطارات FFmpeg البصري اختياري وغير مفعّل افتراضيًا لأنه يزيد زمن CPU ولا يُعد نموذج AI. اختبر T4 الحقيقي واحفظ سجل GPU والنتيجة قبل إعلان توافقه.